# Семинар по Мультиагентным Системам

**Продолжение семинара "AI Agents"**  
**Предварительные знания:** Базовые концепции AI агентов, ReAct паттерн, инструменты

## Обзор семинара

В этом семинаре мы изучим:
1. **Schema Guided Reasoning (SGR)** - структурированное взаимодействие агентов
2. **Паттерны мультиагентных систем** - архитектурные решения
3. **Практическая реализация** - создание системы на SmolAgents




In [ ]:
import os

os.environ["OPENROUTER_API_KEY"]="Your key here"

In [14]:
# Настройка и импорты


from typing import List, Dict, Union, Optional, Literal, Any
from pydantic import BaseModel, Field
from annotated_types import MinLen, MaxLen, Ge, Le
import json

# Настройка OpenRouter API
# Получите API ключ на https://openrouter.ai/keys
# Установите: export OPENROUTER_API_KEY="your-key-here"

USE_REAL_LLM = False  # Флаг для переключения между моками и реальными LLM

if "OPENROUTER_API_KEY" in os.environ:
    try:
        from openai import OpenAI
        
        # Создаем клиент для OpenRouter
        openrouter_client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ["OPENROUTER_API_KEY"],
        )
        
        # Выбираем модель (можно изменить на другую)
        # Доступные модели: https://openrouter.ai/models
        DEFAULT_MODEL = "qwen/qwen3-30b-a3b-instruct-2507"  # 
        USE_REAL_LLM = True
        print(f"OpenRouter API настроен!")
        print(f"Используемая модель: {DEFAULT_MODEL}")
        print(f"Агенты будут использовать реальные LLM вызовы")
    except ImportError:
        print("Установите OpenAI библиотеку: pip install openai")
        print("Используем моки для демонстрации")
        USE_REAL_LLM = False
else:
    print("OPENROUTER_API_KEY не найден")
    print("Для использования реальных LLM:")
    print("  1. Получите ключ на https://openrouter.ai/keys")
    print("  2. export OPENROUTER_API_KEY='your-key'")
    print("  3. Перезапустите notebook")
    print("\nИспользуем моки для демонстрации")

print("\nСеминар по мультиагентным системам начинается!")


OpenRouter API настроен!
Используемая модель: qwen/qwen3-30b-a3b-instruct-2507
Агенты будут использовать реальные LLM вызовы

Семинар по мультиагентным системам начинается!


In [ ]:
# Helper функция для вызова LLM с structured output через OpenRouter

def call_llm_with_schema(
    prompt: str, 
    response_schema: BaseModel, 
    system_prompt: Optional[str] = None
) -> BaseModel:
    """
    Вызывает LLM через OpenRouter с ограничением выхода схемой Pydantic
    
    Args:
        prompt: Промпт для LLM
        response_schema: Pydantic модель для structured output
        system_prompt: Системный промпт (опционально)
    
    Returns:
        Экземпляр response_schema с заполненными данными
    """
    if not USE_REAL_LLM:
        # Если нет API ключа, возвращаем None (будем использовать моки)
        return None
    
    try:
        messages = []
        
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        
        messages.append({"role": "user", "content": prompt})
        
        # OpenRouter поддерживает structured outputs через параметр response_format
        # ВАЖНО: Не все модели поддерживают structured outputs!
        # Claude 3.5 Sonnet, GPT-4o, GPT-4o-mini поддерживают
        
        # Для OpenRouter нужно использовать JSON schema
        schema_dict = response_schema.model_json_schema()
        
        response = openrouter_client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=messages,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": response_schema.__name__,
                    "schema": schema_dict,
                    "strict": True
                }
            },
            temperature=0.7,
            max_tokens=2000
        )
        
        # Парсим JSON ответ в Pydantic модель
        json_response = response.choices[0].message.content
        return response_schema.model_validate_json(json_response)
        
    except Exception as e:
        print(f"Ошибка при вызове LLM: {e}")
        print(f"Переключаемся на моки...")
        return None

print("Helper функция для OpenRouter API создана!")
print("Функция call_llm_with_schema() готова к использованию")


Helper функция для OpenRouter API создана!
Функция call_llm_with_schema() готова к использованию


## Секция 1: От одиночных агентов к мультиагентным системам (15 мин)

### 1.1 Ограничения одиночных агентов

В базовом семинаре по агентам мы изучили ReAct паттерн и создание агентов с инструментами. Но одиночные агенты имеют ограничения:

- **Сложность задач** превышает возможности одного агента
- **Специализация vs универсальность** - сложно быть экспертом во всем
- **Производительность** - нет параллелизации
- **Отказоустойчивость** - одна точка отказа

### 1.2 Что такое мультиагентная система?

**Мультиагентная система (МАС)** - это система из множества автономных агентов, которые:
- Имеют **специализированные роли**
- **Взаимодействуют** для решения сложных задач  
- **Координируют** свои действия
- Работают **параллельно** и **асинхронно**

### 1.3 Архитектуры мультиагентных систем

```
Централизованная:           Децентрализованная:         Иерархическая:
                           
    Координатор                Agent A ←→ Agent B            Supervisor
   ↙ ↓ ↓ ↘                        ↗ ↘   ↙ ↗                ↙        ↘
Agent Agent Agent Agent      Agent C ←→ Agent D        Team A      Team B
  A    B    C    D                                   ↙  ↘      ↙    ↘
                                                 Ag1  Ag2   Ag3   Ag4
```

**Ключевые преимущества:**
-  Специализация агентов на конкретных задачах
-  Параллельная обработка различных аспектов
-  Отказоустойчивость через резервирование
-  Модульность и переиспользование компонентов


In [13]:
# Пример: Сравнение подходов

print("=== ОДИНОЧНЫЙ АГЕНТ ===")
single_agent_approach = """
Задача: Создать презентацию о новой технологии

Один универсальный агент должен:
1. Исследовать технологию
2. Проанализировать рынок  
3. Создать содержание
4. Оформить презентацию
5. Проверить качество

Проблемы:
 Поверхностная экспертиза в каждой области
 Последовательное выполнение (медленно)
 Сложно оптимизировать под каждую задачу
 Ошибка в одном месте влияет на все
"""

print("=== МУЛЬТИАГЕНТНАЯ СИСТЕМА ===")
multiagent_approach = """
Задача: Создать презентацию о новой технологии

Специализированные агенты:
- Research Agent: Глубокое исследование технологии
- Market Agent: Анализ рынка и конкурентов
- Content Agent: Создание структурированного контента
- Design Agent: Профессиональное оформление
- QA Agent: Контроль качества и улучшения

Преимущества:
 Глубокая экспертиза в каждой области
 Параллельное выполнение (быстрее)
 Каждый агент оптимизирован под свою задачу
 Отказ одного агента не блокирует систему
"""

print(single_agent_approach)
print(multiagent_approach)

# Визуализация разницы во времени
import time
def simulate_time(name, duration):
    print(f"⏱ {name}: {duration} секунд")

print("\n Сравнение времени выполнения:")
print("\nОдиночный агент (последовательно):")
total_time = 0
for task, time_needed in [("Исследование", 30), ("Анализ", 25), ("Содержание", 20), ("Оформление", 15), ("Проверка", 10)]:
    simulate_time(task, time_needed)
    total_time += time_needed
print(f" Общее время: {total_time} секунд")

print("\nМультиагентная система (параллельно):")
# Параллельные фазы
phase1_time = max(30, 25)  # Research и Market параллельно
phase2_time = max(20, 15)  # Content и Design параллельно  
phase3_time = 10           # QA последовательно
total_ma_time = phase1_time + phase2_time + phase3_time

print(f"⏱ Фаза 1 (Research + Market): {phase1_time} сек")
print(f"⏱ Фаза 2 (Content + Design): {phase2_time} сек") 
print(f"⏱ Фаза 3 (QA): {phase3_time} сек")
print(f" Общее время: {total_ma_time} секунд")

print(f"\n Ускорение: {total_time/total_ma_time:.1f}x быстрее!")


=== ОДИНОЧНЫЙ АГЕНТ ===
=== МУЛЬТИАГЕНТНАЯ СИСТЕМА ===

Задача: Создать презентацию о новой технологии

Один универсальный агент должен:
1. Исследовать технологию
2. Проанализировать рынок  
3. Создать содержание
4. Оформить презентацию
5. Проверить качество

Проблемы:
 Поверхностная экспертиза в каждой области
 Последовательное выполнение (медленно)
 Сложно оптимизировать под каждую задачу
 Ошибка в одном месте влияет на все


Задача: Создать презентацию о новой технологии

Специализированные агенты:
- Research Agent: Глубокое исследование технологии
- Market Agent: Анализ рынка и конкурентов
- Content Agent: Создание структурированного контента
- Design Agent: Профессиональное оформление
- QA Agent: Контроль качества и улучшения

Преимущества:
 Глубокая экспертиза в каждой области
 Параллельное выполнение (быстрее)
 Каждый агент оптимизирован под свою задачу
 Отказ одного агента не блокирует систему


 Сравнение времени выполнения:

Одиночный агент (последовательно):
⏱ Исследование: 

## Секция 2: Schema Guided Reasoning (SGR) (20 мин)

### 2.1 Что такое SGR?

**Schema Guided Reasoning** - это метод направления рассуждений LLM через предопределенные структурированные схемы.

**Основные компоненты:**
- **Schema** - структурированное представление знаний о задаче, ролях и взаимодействиях
- **Guided Reasoning** - использование схем для направления процесса рассуждений
- **Constrained Generation** - ограничение генерации модели в рамках схемы

**Применение в мультиагентных системах:**
-  Стандартизация коммуникации между агентами
-  Обеспечение структурированных ответов 
-  Предсказуемость взаимодействий
-  Контроль качества результатов

**Источник:** [Schema-Guided Reasoning Patterns by Abdullin](https://abdullin.com/schema-guided-reasoning/patterns)

### 2.2 Три основных SGR паттерна

####  Паттерн 1: Cascade (Каскад)
**Назначение:** Принуждение LLM следовать предопределенным шагам рассуждения

**Принцип:** Каждый шаг выделяет "мыслительный бюджет" для продвижения рассуждения на один шаг дальше

**Пример использования:**


In [15]:
# SGR Паттерн 1: Cascade (Каскад)

from pydantic import BaseModel
from annotated_types import Ge, Le
from typing import List, Annotated

class CandidateEvaluation(BaseModel):
    """
    Cascade паттерн: каждое поле направляет рассуждение на следующий шаг
    """
    # Шаг 1: Сначала обобщение знаний о кандидате
    brief_candidate_summary: str
    
    # Шаг 2: Затем оценка соответствия навыков от 1 до 10
    rate_skill_match: Annotated[int, Ge(1), Le(10)]
    
    # Шаг 3: И только потом финальное решение
    final_recommendation: Literal["hire", "reject", "hold"]

# Демонстрация схемы
sample_evaluation = CandidateEvaluation(
    brief_candidate_summary="Опытный Python разработчик с 5+ годами опыта в ML и веб-разработке. Хорошие навыки коммуникации.",
    rate_skill_match=8,
    final_recommendation="hire"
)

print("Пример Cascade паттерна:")
print(f"Сводка: {sample_evaluation.brief_candidate_summary}")
print(f"Оценка навыков: {sample_evaluation.rate_skill_match}/10")  
print(f"Решение: {sample_evaluation.final_recommendation}")

print("\n Cascade обеспечивает логическую последовательность: обобщение → оценка → решение")


Пример Cascade паттерна:
Сводка: Опытный Python разработчик с 5+ годами опыта в ML и веб-разработке. Хорошие навыки коммуникации.
Оценка навыков: 8/10
Решение: hire

 Cascade обеспечивает логическую последовательность: обобщение → оценка → решение


In [18]:
# SGR Паттерн 2: Routing (Маршрутизация)

from typing import Union

# Различные типы задач для техподдержки
class HardwareIssue(BaseModel):
    kind: Literal["hardware"]
    component: Literal["battery", "display", "keyboard", "motherboard"]
    urgency: Literal["low", "medium", "high"]

class SoftwareIssue(BaseModel):
    kind: Literal["software"]
    software_name: str
    error_type: Literal["crash", "performance", "bug", "compatibility"]
    
class NetworkIssue(BaseModel):
    kind: Literal["network"]
    connection_type: Literal["wifi", "ethernet", "vpn"]
    symptoms: List[str]

class SupportTriage(BaseModel):
    """
    Routing паттерн: принуждает LLM выбрать ОДИН путь из многих
    """
    # LLM должен выбрать конкретный тип проблемы
    issue: Union[HardwareIssue, SoftwareIssue, NetworkIssue]
    
    # Дополнительная информация
    priority: Literal["low", "normal", "high", "critical"]
    estimated_resolution_time: int  # в минутах

# Примеры различных маршрутов
examples = [
    SupportTriage(
        issue=HardwareIssue(kind="hardware", component="display", urgency="high"),
        priority="high",
        estimated_resolution_time=30
    ),
    SupportTriage(
        issue=SoftwareIssue(kind="software", software_name="Microsoft Word", error_type="crash"),
        priority="normal", 
        estimated_resolution_time=15
    ),
    SupportTriage(
        issue=NetworkIssue(kind="network", connection_type="wifi", symptoms=["slow_speed", "frequent_disconnects"]),
        priority="normal",
        estimated_resolution_time=45
    )
]

print("Примеры Routing паттерна:")
for i, example in enumerate(examples, 1):
    print(f"\n{i} Тип проблемы: {example.issue.kind}")
    print(f"Детали: {example.issue.dict()}")
    print(f"Приоритет: {example.priority}")
    print(f"Время решения: {example.estimated_resolution_time} мин")

print("\n Routing заставляет LLM выбрать конкретный путь и заполнить соответствующие поля")


Примеры Routing паттерна:

1 Тип проблемы: hardware
Детали: {'kind': 'hardware', 'component': 'display', 'urgency': 'high'}
Приоритет: high
Время решения: 30 мин

2 Тип проблемы: software
Детали: {'kind': 'software', 'software_name': 'Microsoft Word', 'error_type': 'crash'}
Приоритет: normal
Время решения: 15 мин

3 Тип проблемы: network
Детали: {'kind': 'network', 'connection_type': 'wifi', 'symptoms': ['slow_speed', 'frequent_disconnects']}
Приоритет: normal
Время решения: 45 мин

 Routing заставляет LLM выбрать конкретный путь и заполнить соответствующие поля


/var/folders/6c/j5sghwf51gx41g_055zlcs_m0000gq/T/ipykernel_19030/4030268221.py:54: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"Детали: {example.issue.dict()}")


In [19]:
# SGR Паттерн 3: Cycle (Цикл)

class RiskFactor(BaseModel):
    explanation: str
    severity: Literal["low", "medium", "high", "critical"]
    mitigation_strategy: str

class RiskAssessment(BaseModel):
    """
    Cycle паттерн: принуждает LLM повторить шаги рассуждения несколько раз
    """
    # Минимум 2, максимум 5 факторов риска
    factors: Annotated[List[RiskFactor], MinLen(2), MaxLen(5)]
    
    # Общая оценка после анализа всех факторов
    overall_risk_level: Literal["low", "medium", "high", "critical"]
    immediate_actions_needed: bool

# Пример анализа рисков
sample_assessment = RiskAssessment(
    factors=[
        RiskFactor(
            explanation="Серверная комната имеет плохую вентиляцию, что может привести к перегреву оборудования",
            severity="high",
            mitigation_strategy="Установить дополнительные системы кондиционирования"
        ),
        RiskFactor(
            explanation="Устаревшие сетевые сетевые устройства без обновлений безопасности", 
            severity="critical",
            mitigation_strategy="Запланировать замену в течение 30 дней"
        ),
        RiskFactor(
            explanation="Отсутствует резервное копирование базы данных",
            severity="high", 
            mitigation_strategy="Настроить ежедневные автоматические бэкапы"
        )
    ],
    overall_risk_level="critical",
    immediate_actions_needed=True
)

print("Пример Cycle паттерна:")
print(f"Найдено факторов риска: {len(sample_assessment.factors)}")
for i, factor in enumerate(sample_assessment.factors, 1):
    print(f"\n{i} Риск: {factor.explanation}")
    print(f"   Серьезность: {factor.severity}")
    print(f"   Решение: {factor.mitigation_strategy}")

print(f"\n Общий уровень риска: {sample_assessment.overall_risk_level}")
print(f" Немедленные действия нужны: {'Да' if sample_assessment.immediate_actions_needed else 'Нет'}")

print("\n Cycle заставляет LLM создать несколько экземпляров одного типа объекта")


Пример Cycle паттерна:
Найдено факторов риска: 3

1 Риск: Серверная комната имеет плохую вентиляцию, что может привести к перегреву оборудования
   Серьезность: high
   Решение: Установить дополнительные системы кондиционирования

2 Риск: Устаревшие сетевые сетевые устройства без обновлений безопасности
   Серьезность: critical
   Решение: Запланировать замену в течение 30 дней

3 Риск: Отсутствует резервное копирование базы данных
   Серьезность: high
   Решение: Настроить ежедневные автоматические бэкапы

 Общий уровень риска: critical
 Немедленные действия нужны: Да

 Cycle заставляет LLM создать несколько экземпляров одного типа объекта


### 2.3 Применение SGR в мультиагентных системах

#### Как SGR помогает в координации агентов:

**1. Стандартизация коммуникации**
- Каждый агент знает, в каком формате ожидать данные
- Уменьшается количество ошибок парсинга
- Упрощается отладка взаимодействий

**2. Предсказуемость поведения**
- Cascade обеспечивает логическую последовательность действий агента
- Routing гарантирует выбор правильного следующего агента
- Cycle создает полноценные наборы данных

**3. Контроль качества**
- Схемы валидируют выходные данные агентов
- Ограничения (Ge, Le, MinLen, MaxLen) предотвращают некорректные значения
- Типизация обеспечивает консистентность данных

#### Пример схемы коммуникации между агентами:


In [20]:
# Схема коммуникации между агентами

from datetime import datetime
from typing import Dict, Any

# Определяем роли агентов в системе
class AgentRole(BaseModel):
    agent_id: str
    role_type: Literal["researcher", "analyzer", "creator", "reviewer", "coordinator"]
    capabilities: List[str]
    max_concurrent_tasks: int

# Схема для передачи сообщений между агентами
class AgentMessage(BaseModel):
    """
    Универсальная схема для коммуникации агентов
    Использует все три SGR паттерна
    """
    # Cascade: Структурированный порядок обработки
    sender_id: str                          # Сначала - кто отправил
    task_context: str                       # Затем - контекст задачи
    data_payload: Dict[str, Any]            # Потом - данные
    processing_instructions: List[str]      # И наконец - инструкции
    
    # Routing: Выбор получателя
    recipient: Union[
        str,                                # Конкретный агент по ID
        Literal["broadcast_all"],           # Всем агентам
        Literal["next_available"]           # Следующему доступному
    ]
    
    # Цикл может быть встроен в processing_instructions
    priority: Literal["low", "normal", "high", "urgent"]
    timestamp: datetime = Field(default_factory=datetime.now)
    expected_response_time: int = Field(ge=1, le=3600)  # секунды

# Примеры сообщений в мультиагентной системе
messages = [
    AgentMessage(
        sender_id="coordinator_001",
        task_context="Создание учебного материала по 'Машинному обучению'",
        data_payload={
            "topic": "Машинное обучение", 
            "target_level": "beginner",
            "required_sections": ["introduction", "algorithms", "examples"]
        },
        processing_instructions=[
            "Найти актуальную информацию по теме",
            "Структурировать в понятном формате",
            "Добавить практические примеры"
        ],
        recipient="researcher_001", 
        priority="normal",
        expected_response_time=300
    ),
    
    AgentMessage(
        sender_id="researcher_001",
        task_context="Результаты исследования по машинному обучению", 
        data_payload={
            "research_summary": "ML - область ИИ, позволяющая компьютерам обучаться...",
            "key_concepts": ["supervised learning", "unsupervised learning", "neural networks"],
            "information_sources": ["scikit-learn.org", "tensorflow.org", "coursera ML course"],
            "confidence_score": 9
        },
        processing_instructions=[
            "Проанализировать сложность материала",
            "Определить целевую аудиторию", 
            "Оценить время изучения"
        ],
        recipient="analyzer_001",
        priority="normal", 
        expected_response_time=180
    )
]

print("Примеры структурированной коммуникации агентов:")
for i, msg in enumerate(messages, 1):
    print(f"\n Сообщение {i}:")
    print(f"   От: {msg.sender_id} → Кому: {msg.recipient}")
    print(f"   Контекст: {msg.task_context}")
    print(f"   Приоритет: {msg.priority}")
    print(f"   Время ответа: {msg.expected_response_time} сек")
    print(f"   Инструкций: {len(msg.processing_instructions)}")

print("\n SGR схемы обеспечивают четкую структуру взаимодействия агентов")


Примеры структурированной коммуникации агентов:

 Сообщение 1:
   От: coordinator_001 → Кому: researcher_001
   Контекст: Создание учебного материала по 'Машинному обучению'
   Приоритет: normal
   Время ответа: 300 сек
   Инструкций: 3

 Сообщение 2:
   От: researcher_001 → Кому: analyzer_001
   Контекст: Результаты исследования по машинному обучению
   Приоритет: normal
   Время ответа: 180 сек
   Инструкций: 3

 SGR схемы обеспечивают четкую структуру взаимодействия агентов


---

## ЗАДАНИЕ 1: Создание SGR схемы для нового агента

**Описание задачи:**
Создайте Pydantic схему для агента-переводчика (Translator Agent), который переводит учебные материалы на другой язык. Используйте **Cascade паттерн** для структурирования работы агента.

**Требования:**
1. Агент должен сначала проанализировать исходный текст
2. Затем определить целевую аудиторию для перевода
3. Выполнить перевод
4. Проверить качество перевода (оценка от 1 до 10)
5. Финальный статус перевода: "ready", "needs_review", "poor_quality"

**Подсказка:** Используйте `Annotated[int, Ge(1), Le(10)]` для ограничения оценки качества.

**Время выполнения:** 10-15 минут


In [24]:
# РЕШЕНИЕ ЗАДАНИЯ 1

class TranslatorOutput(BaseModel):
    """
    Cascade паттерн для агента-переводчика
    Шаги выполняются последовательно, каждый опирается на предыдущий
    """
    # Шаг 1: Анализ исходного текста
    source_text_analysis: str = Field(
        description="Анализ структуры, сложности и ключевых терминов исходного текста"
    )
    
    # Шаг 2: Определение целевой аудитории
    target_audience: Literal["beginners", "professionals", "academics", "general"] = Field(
        description="Целевая аудитория для адаптации стиля перевода"
    )
    
    # Шаг 3: Выполнение перевода
    translated_text: str = Field(
        description="Переведенный текст на целевом языке"
    )
    
    # Шаг 4: Оценка качества перевода
    translation_quality_score: Annotated[int, Ge(1), Le(10)] = Field(
        description="Оценка качества перевода от 1 до 10"
    )
    
    # Шаг 5: Финальный статус
    final_status: Literal["ready", "needs_review", "poor_quality"] = Field(
        description="Статус готовности перевода"
    )
    
    # Дополнительная информация
    terminology_notes: Optional[List[str]] = Field(
        default=None,
        description="Замечания по специфичной терминологии"
    )

# Тестируем с реальным LLM или моком
print("=" * 60)
print("ЗАДАНИЕ 1: Тест Translator Agent")
print("=" * 60)

test_text = """
Машинное обучение является подразделом искусственного интеллекта, которое 
фокусируется на создании систем, способных обучаться на данных.
"""


    
prompt = f"""Переведите следующий текст на английский язык:

Исходный текст: {test_text}

Целевой язык: Английский
Целевая аудитория: Профессионалы в области AI/ML

Следуйте cascade паттерну:
1. Сначала проанализируйте текст (структура, сложность, термины)
2. Определите целевую аудиторию
3. Выполните качественный перевод
4. Оцените качество перевода (1-10)
5. Определите финальный статус"""
    
translation_result = call_llm_with_schema(
    prompt=prompt,
    response_schema=TranslatorOutput,
    system_prompt="Вы - профессиональный технический переводчик с опытом в AI/ML."
)

if translation_result:
    print("\nРЕЗУЛЬТАТ ОТ LLM:")
    print(f"Анализ: {translation_result.source_text_analysis}")
    print(f"Аудитория: {translation_result.target_audience}")
    print(f"\nПеревод:\n{translation_result.translated_text}")
    print(f"Качество: {translation_result.translation_quality_score}/10")
    print(f"Статус: {translation_result.final_status}")
else:
    translation_result = None

if translation_result is None:
    print("\nИспользуем МОК для демонстрации...")
    translation_result = TranslatorOutput(
        source_text_analysis="Технический текст средней сложности с терминологией из области ML",
        target_audience="professionals",
        translated_text="Machine learning is a branch of artificial intelligence...",
        translation_quality_score=8,
        final_status="ready",
        terminology_notes=["ML терминология сохранена"]
    )
    print(f"Качество: {translation_result.translation_quality_score}/10")
    print(f"Статус: {translation_result.final_status}")

print("\n" + "=" * 60)
print("Cascade паттерн: анализ -> аудитория -> перевод -> оценка -> статус")


ЗАДАНИЕ 1: Тест Translator Agent

РЕЗУЛЬТАТ ОТ LLM:
Анализ: Исходный текст представляет собой краткое определение машинного обучения как подраздела искусственного интеллекта. Структура предложения простая, текст содержит термины, характерные для области AI/ML, такие как «машинное обучение» и «системы, способные обучаться на данных».
Аудитория: professionals

Перевод:
Machine learning is a subfield of artificial intelligence that focuses on creating systems capable of learning from data.
Качество: 10/10
Статус: ready

Cascade паттерн: анализ -> аудитория -> перевод -> оценка -> статус


## Секция 3: Паттерны мультиагентных систем (25 мин)

Теперь, когда мы знаем как структурировать взаимодействие через SGR, изучим основные паттерны организации мультиагентных систем.

### 3.1 Sequential Pattern (Последовательный)

**Описание:** Агенты работают один за другим, передавая результат следующему
**Когда использовать:** Пайплайн обработки, где выход одного агента - вход другого
**Пример:** Исследование → Анализ → Написание отчета → Проверка качества

```
Agent A → [Result A] → Agent B → [Result B] → Agent C → [Final Result]
```

### 3.2 Parallel Pattern (Параллельный) 

**Описание:** Несколько агентов работают одновременно над разными частями задачи
**Когда использовать:** Независимые подзадачи, которые можно выполнять параллельно  
**Пример:** Анализ разных источников данных одновременно

```
                → Agent A → [Result A]
Input Task      → Agent B → [Result B]  → Aggregator → Final Result
                → Agent C → [Result C]
```

### 3.3 Hierarchical Pattern (Иерархический)

**Описание:** Агенты организованы в иерархию с супервизором и подчиненными
**Когда использовать:** Сложные задачи, требующие координации и планирования
**Пример:** Менеджер проекта → Команды специалистов → Исполнители

```
        Supervisor Agent
       ↙       ↓       ↘
  Team A    Team B    Team C
  ↙  ↘      ↙  ↘      ↙  ↘
Ag1  Ag2  Ag3  Ag4  Ag5  Ag6
```


In [25]:
# Демонстрация паттернов взаимодействия

class TaskResult(BaseModel):
    agent_id: str
    task_type: str 
    result_data: Dict[str, Any]
    processing_time: float
    success: bool

# Симуляция различных агентов
def simulate_agent_work(agent_id: str, task: str, input_data: Any = None) -> TaskResult:
    """Симулирует работу агента"""
    import random
    import time
    
    # Симулируем время обработки
    processing_time = random.uniform(0.1, 0.5)
    
    result_data = {
        "input_received": str(input_data)[:100] if input_data else "None",
        "task_completed": task,
        "output": f"Результат от {agent_id} для задачи: {task}",
        "timestamp": datetime.now().isoformat()
    }
    
    return TaskResult(
        agent_id=agent_id,
        task_type=task,
        result_data=result_data,
        processing_time=processing_time,
        success=True
    )

print(" 1. SEQUENTIAL PATTERN - Последовательная обработка")
print("="*50)

# Sequential pattern: каждый агент использует результат предыдущего
def sequential_processing(initial_task: str):
    print(f"Задача: {initial_task}")
    
    # Этап 1: Исследование
    result1 = simulate_agent_work("researcher", "поиск информации", initial_task)
    print(f"  1⃣ {result1.agent_id}: {result1.result_data['output']}")
    
    # Этап 2: Анализ (использует результат исследования)
    result2 = simulate_agent_work("analyzer", "анализ данных", result1.result_data)
    print(f"  2⃣ {result2.agent_id}: {result2.result_data['output']}")
    
    # Этап 3: Создание контента (использует результат анализа)
    result3 = simulate_agent_work("writer", "написание отчета", result2.result_data) 
    print(f"  3⃣ {result3.agent_id}: {result3.result_data['output']}")
    
    total_time = result1.processing_time + result2.processing_time + result3.processing_time
    print(f"  ⏱ Общее время: {total_time:.2f} сек (последовательно)")
    return total_time

seq_time = sequential_processing("Анализ рынка криптовалют")

print("\n 2. PARALLEL PATTERN - Параллельная обработка")
print("="*50)

def parallel_processing(initial_task: str):
    print(f"Задача: {initial_task}")
    
    # Параллельно запускаем нескольких агентов
    import concurrent.futures
    import threading
    
    tasks = [
        ("researcher", "поиск в научных источниках"),
        ("market_analyst", "анализ рыночных трендов"), 
        ("tech_expert", "технический анализ"),
    ]
    
    results = []
    start_time = datetime.now()
    
    # Симулируем параллельное выполнение
    for agent_id, task in tasks:
        result = simulate_agent_work(agent_id, task, initial_task)
        results.append(result)
        print(f"   {result.agent_id}: {result.result_data['output']}")
    
    # В реальности это время было бы max() от всех времен выполнения
    parallel_time = max(result.processing_time for result in results)
    print(f"  ⏱ Общее время: {parallel_time:.2f} сек (параллельно)")
    
    # Агрегация результатов
    aggregated = simulate_agent_work("aggregator", "объединение результатов", 
                                   [r.result_data for r in results])
    print(f"   {aggregated.agent_id}: {aggregated.result_data['output']}")
    
    total_parallel_time = parallel_time + aggregated.processing_time
    print(f"  ⏱ Общее время с агрегацией: {total_parallel_time:.2f} сек")
    return total_parallel_time

par_time = parallel_processing("Анализ рынка криптовалют")

print(f"\n Ускорение при параллелизации: {seq_time/par_time:.1f}x")

print("\n 3. HIERARCHICAL PATTERN - Иерархическая координация")
print("="*50)

def hierarchical_processing(initial_task: str):
    print(f"Задача: {initial_task}")
    
    # Супервизор планирует работу
    supervisor_plan = simulate_agent_work("supervisor", "планирование задач", initial_task)
    print(f"   Supervisor: {supervisor_plan.result_data['output']}")
    
    # Команды работают параллельно под руководством супервизора
    teams = {
        "research_team": ["researcher_1", "researcher_2"],
        "analysis_team": ["analyst_1", "analyst_2"], 
        "content_team": ["writer_1", "designer_1"]
    }
    
    team_results = {}
    for team_name, members in teams.items():
        print(f"\n   Команда {team_name}:")
        team_output = []
        for member in members:
            result = simulate_agent_work(member, f"работа в {team_name}", supervisor_plan.result_data)
            print(f"     {result.agent_id}: {result.result_data['output']}")
            team_output.append(result)
        team_results[team_name] = team_output
    
    # Супервизор координирует финальную сборку
    final_result = simulate_agent_work("supervisor", "финальная координация", team_results)
    print(f"\n   Финальная координация: {final_result.result_data['output']}")

hierarchical_processing("Анализ рынка криптовалют")


 1. SEQUENTIAL PATTERN - Последовательная обработка
Задача: Анализ рынка криптовалют
  1⃣ researcher: Результат от researcher для задачи: поиск информации
  2⃣ analyzer: Результат от analyzer для задачи: анализ данных
  3⃣ writer: Результат от writer для задачи: написание отчета
  ⏱ Общее время: 0.78 сек (последовательно)

 2. PARALLEL PATTERN - Параллельная обработка
Задача: Анализ рынка криптовалют
   researcher: Результат от researcher для задачи: поиск в научных источниках
   market_analyst: Результат от market_analyst для задачи: анализ рыночных трендов
   tech_expert: Результат от tech_expert для задачи: технический анализ
  ⏱ Общее время: 0.47 сек (параллельно)
   aggregator: Результат от aggregator для задачи: объединение результатов
  ⏱ Общее время с агрегацией: 0.79 сек

 Ускорение при параллелизации: 1.0x

 3. HIERARCHICAL PATTERN - Иерархическая координация
Задача: Анализ рынка криптовалют
   Supervisor: Результат от supervisor для задачи: планирование задач

   Команда res

---

## ЗАДАНИЕ 2: Реализация Debate Pattern

**Описание задачи:**
Реализуйте Debate Pattern для принятия решения о выборе технологии. Создайте двух агентов с противоположными точками зрения, которые будут обсуждать выбор между двумя технологиями.

**Требования:**
1. Создайте класс `DebateAgent` с методом `argue(topic, position)`
2. Первый агент выступает "за" технологию A
3. Второй агент выступает "за" технологию B
4. Каждый агент приводит 3 аргумента
5. Создайте функцию `run_debate()`, которая запускает дебаты и выводит аргументы обеих сторон
6. Определите "победителя" на основе силы аргументов (можно использовать простую оценку)

**Подсказка:** Используйте моки для симуляции аргументов агентов.

**Время выполнения:** 15-20 минут


In [ ]:
# РЕШЕНИЕ ЗАДАНИЯ 2

class Argument(BaseModel):
    """Структура для одного аргумента"""
    point: str
    evidence: str
    strength: Annotated[int, Ge(1), Le(10)]  # Сила аргумента

class DebateArguments(BaseModel):
    """SGR Cycle: множественные аргументы для дебатов"""
    arguments: Annotated[List[Argument], MinLen(2), MaxLen(5)] = Field(
        description="Список аргументов в поддержку позиции"
    )
    total_strength: int = Field(description="Суммарная сила всех аргументов")

class DebateAgent:
    """Агент для участия в дебатах"""
    
    def __init__(self, agent_id: str, position: str):
        self.agent_id = agent_id
        self.position = position
        
    def argue(self, topic: str, technology: str) -> List[Argument]:
        """Генерирует аргументы в поддержку своей позиции"""
        print(f"\n[{self.agent_id}] Аргументы в пользу {technology}:")
        
        # Пытаемся использовать LLM если доступен
        
        llm_arguments = self._argue_with_llm(topic, technology)
        if llm_arguments:
            print(f"   [Используем LLM для генерации аргументов]")
            for i, arg in enumerate(llm_arguments.arguments, 1):
                print(f"  {i}. {arg.point}")
                print(f"     Обоснование: {arg.evidence}")
                print(f"     Сила аргумента: {arg.strength}/10")
            return llm_arguments.arguments
        
        # Без моков: только реальный LLM
        raise RuntimeError("OPENROUTER_API_KEY is required: DebateAgent работает только с реальным LLM.")
    
    def _argue_with_llm(self, topic: str, technology: str) -> Optional[DebateArguments]:
        """Генерирует аргументы через LLM"""
        try:
            prompt = f"""Вы - эксперт-адвокат, защищающий использование технологии {technology} для задачи: {topic}

Создайте 3-4 убедительных аргумента в пользу {technology}:
- Каждый аргумент должен иметь четкую точку (point)
- Подкрепляйте каждый аргумент конкретными доказательствами (evidence)
- Оцените силу каждого аргумента от 1 до 10 (strength)
- Аргументы должны быть реалистичными и обоснованными
- Фокусируйтесь на преимуществах {technology} для данной задачи

Будьте убедительны, но объективны."""

            result = call_llm_with_schema(
                prompt=prompt,
                response_schema=DebateArguments,
                system_prompt=f"Вы - опытный технический консультант, специализирующийся на {technology}. Вы умеете создавать убедительные, но честные аргументы."
            )
            
            if result:
                result.total_strength = sum(arg.strength for arg in result.arguments)
                return result
        except Exception as e:
            print(f"   Ошибка LLM: {e}, используем моки...")
        
        return None
    
    # def _argue_with_mocks(self, technology: str) -> List[Argument]:
    #     """Генерирует аргументы через моки"""
    #     # Моки аргументов для разных технологий
    #     if technology == "PostgreSQL":
    #         arguments = [
    #             Argument(
    #                 point="ACID-совместимость",
    #                 evidence="PostgreSQL полностью поддерживает ACID транзакции, что критично для финансовых приложений",
    #                 strength=9
    #             ),
    #             Argument(
    #                 point="Расширенные типы данных",
    #                 evidence="Поддержка JSON, массивов, геоданных из коробки",
    #                 strength=8
    #             ),
    #             Argument(
    #                 point="Зрелость и стабильность",
    #                 evidence="30+ лет разработки, огромное комьюнити, проверенная надежность",
    #                 strength=9
    #             )
    #         ]
    #     else:  # MongoDB
    #         arguments = [
    #             Argument(
    #                 point="Гибкая схема данных",
    #                 evidence="NoSQL позволяет быстро итерировать и менять структуру данных без миграций",
    #                 strength=8
    #             ),
    #             Argument(
    #                 point="Горизонтальное масштабирование",
    #                 evidence="Встроенный шардинг позволяет легко масштабировать на множество серверов",
    #                 strength=9
    #             ),
    #             Argument(
    #                 point="Производительность для чтения",
    #                 evidence="Оптимизирована для высоких нагрузок по чтению данных",
    #                 strength=7
    #             )
    #         ]
        
    #     return arguments

def run_debate(topic: str, tech_a: str, tech_b: str):
    """
    Запускает дебаты между двумя агентами
    Демонстрация Debate Pattern
    """
    print("=" * 60)
    print(f"ДЕБАТЫ: {topic}")
    print(f"Технология A: {tech_a} vs Технология B: {tech_b}")
    print("=" * 60)
    
    # Создаем агентов с противоположными позициями
    agent_a = DebateAgent("Agent_Pro_A", f"за {tech_a}")
    agent_b = DebateAgent("Agent_Pro_B", f"за {tech_b}")
    
    # Каждый агент приводит свои аргументы
    arguments_a = agent_a.argue(topic, tech_a)
    arguments_b = agent_b.argue(topic, tech_b)
    
    # Подсчитываем общую силу аргументов
    total_strength_a = sum(arg.strength for arg in arguments_a)
    total_strength_b = sum(arg.strength for arg in arguments_b)
    
    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТЫ ДЕБАТОВ:")
    print(f"  {tech_a}: {total_strength_a} баллов")
    print(f"  {tech_b}: {total_strength_b} баллов")
    
    # Определяем победителя
    if total_strength_a > total_strength_b:
        winner = tech_a
        margin = total_strength_a - total_strength_b
    elif total_strength_b > total_strength_a:
        winner = tech_b
        margin = total_strength_b - total_strength_a
    else:
        winner = "Ничья"
        margin = 0
    
    if winner != "Ничья":
        print(f"\nПобедитель: {winner} (преимущество: {margin} баллов)")
    else:
        print(f"\nРезультат: {winner}")
    
    print("=" * 60)
    
    return {
        "winner": winner,
        "scores": {tech_a: total_strength_a, tech_b: total_strength_b},
        "margin": margin
    }

# Запускаем дебаты
result = run_debate(
    topic="Выбор базы данных для e-commerce приложения",
    tech_a="PostgreSQL",
    tech_b="MongoDB"
)

print("\n" + "=" * 60)
print("Debate Pattern продемонстрирован!")
if USE_REAL_LLM:
    print("Использовались РЕАЛЬНЫЕ LLM для генерации аргументов")
    print("Каждый агент создал уникальные аргументы через OpenRouter API")
else:
    print("Использовались МОКИ для демонстрации")
    print("Для реальных LLM установите OPENROUTER_API_KEY")
print("Два агента с разными точками зрения помогают принять взвешенное решение")
print("=" * 60)


ДЕБАТЫ: Выбор базы данных для e-commerce приложения
Технология A: PostgreSQL vs Технология B: MongoDB

[Agent_Pro_A] Аргументы в пользу PostgreSQL:
   [Используем LLM для генерации аргументов]
  1. Поддержка сложных запросов и транзакций
     Обоснование: PostgreSQL обеспечивает мощные возможности для выполнения сложных запросов, что критически важно для e-commerce, где часто требуется работа с большим количеством связанных данных, таких как товары, заказы, пользователи и отзывы. Благодаря поддержке транзакций ACID, PostgreSQL гарантирует целостность данных при одновременном выполнении множества операций. Это особенно важно в сценариях, где несколько пользователей могут одновременно добавлять товары в корзину или оформлять заказы.
     Сила аргумента: 9/10
  2. Гибкость и расширяемость
     Обоснование: PostgreSQL предлагает высокую гибкость за счет поддержки множества типов данных, включая JSON и XML, что позволяет легко интегрировать с различными внешними API и сервисами. Кроме того,

### 3.4 Дополнительные паттерны

#### Debate/Competition Pattern (Дебаты/Соревнование)
- **Описание:** Агенты с разными точками зрения обсуждают проблему
- **Применение:** Принятие решений, творческие задачи, проверка качества
- **Пример:** Разные эксперты оценивают бизнес-план

#### Reflection Pattern (Рефлексия)
- **Описание:** Один агент создает контент, другой его критикует и улучшает
- **Применение:** Итеративное улучшение качества результата
- **Пример:** Писатель → Редактор → Улучшенный текст

#### Round-Robin Pattern (По очереди)
- **Описание:** Агенты по очереди обрабатывают задачи из общей очереди
- **Применение:** Балансировка нагрузки, обработка потока однотипных задач
- **Пример:** Служба поддержки клиентов

---

## Секция 4: Коммуникация и координация (15 мин)

### 4.1 Механизмы коммуникации в мультиагентных системах

**Прямая передача сообщений:** Агент-агент
- Простейший механизм
- Подходит для небольших систем
- Может привести к сложности при масштабировании

**Брокер сообщений:** Централизованная очередь
- Все сообщения проходят через центральный узел
- Упрощает отслеживание и отладку
- Единая точка отказа

**Shared Memory:** Общая память состояния
- Агенты читают/записывают в общую структуру данных
- Быстрый доступ к данным
- Требует синхронизации

**Event-driven:** Система событий
- Агенты подписываются на события
- Слабая связанность
- Асинхронная обработка


## Секция 5: Практическая реализация - "Умная система создания учебных материалов" (35 мин)

### 5.1 Постановка задачи

**Цель:** Создать мультиагентную систему для автоматического создания качественных учебных материалов

**Входные данные:** Тема для изучения (например, "Blockchain технологии", "Машинное обучение") 

**Выходной результат:** Комплект учебных материалов:
-  Структурированный конспект с ключевыми понятиями
-  Практические примеры и кейсы  
-  Набор проверочных вопросов и заданий
-  Рекомендации по дальнейшему изучению
- ⏱ Оценка сложности и времени изучения




In [8]:
# 5.3 SGR схемы для каждого агента

print(" Определяем SGR схемы для всех агентов системы")

# 1. Research Agent - Cascade Pattern
class ResearchOutput(BaseModel):
    """SGR Cascade: пошаговое структурированное исследование"""
    # Шаг 1: Сначала краткое описание темы
    topic_summary: str = Field(description="Краткое описание темы (2-3 предложения)")
    
    # Шаг 2: Затем ключевые концепции
    key_concepts: Annotated[List[str], MinLen(5), MaxLen(15)] = Field(
        description="Основные понятия, которые нужно изучить"
    )
    
    # Шаг 3: Потом признаки сложности
    difficulty_indicators: List[str] = Field(
        description="Признаки, указывающие на сложность темы"
    )
    
    # Шаг 4: Источники информации
    information_sources: Annotated[List[str], MinLen(3)] = Field(
        description="Релевантные источники для изучения"
    )
    
    # Шаг 5: И наконец уверенность в исследовании
    research_confidence: Annotated[int, Ge(1), Le(10)] = Field(
        description="Оценка качества проведенного исследования"
    )

# 2. Complexity Analyzer - Routing Pattern
class BeginnerLevel(BaseModel):
    level: Literal["beginner"]
    estimated_hours: Annotated[int, Ge(2), Le(10)]
    prerequisites: List[str] = []
    learning_approach: str = "Пошаговое изучение с множеством примеров"

class IntermediateLevel(BaseModel):
    level: Literal["intermediate"] 
    estimated_hours: Annotated[int, Ge(8), Le(25)]
    prerequisites: Annotated[List[str], MinLen(1)]
    learning_approach: str = "Теория + практические задачи"
    
class AdvancedLevel(BaseModel):
    level: Literal["advanced"]
    estimated_hours: Annotated[int, Ge(20), Le(100)]
    prerequisites: Annotated[List[str], MinLen(3)]
    learning_approach: str = "Глубокое изучение + исследовательские проекты"

class ComplexityAnalysis(BaseModel):
    """SGR Routing: выбор одного уровня сложности из трех"""
    complexity: Union[BeginnerLevel, IntermediateLevel, AdvancedLevel]
    target_audience: List[str] = Field(description="Для кого предназначен материал")
    learning_path: List[str] = Field(description="Рекомендуемый путь изучения")

# 3. Content Creator - Cascade Pattern
class LearningMaterial(BaseModel):
    """SGR Cascade: структурированное создание контента"""
    # Шаг 1: План
    outline: Annotated[List[str], MinLen(4), MaxLen(8)] = Field(
        description="Структура материала"
    )
    
    # Шаг 2: Детальный контент
    detailed_content: Dict[str, str] = Field(
        description="Секция: детальное содержание"
    )
    
    # Шаг 3: Примеры
    practical_examples: Annotated[List[str], MinLen(2), MaxLen(5)] = Field(
        description="Конкретные примеры для понимания"
    )
    
    # Шаг 4: Ключевые выводы
    key_takeaways: Annotated[List[str], MinLen(3), MaxLen(7)] = Field(
        description="Главные мысли, которые должен запомнить студент"
    )

# 4. Exercise Generator - Cycle Pattern
class Exercise(BaseModel):
    type: Literal["multiple_choice", "coding", "essay", "practical", "analysis"]
    question: str
    difficulty: Literal["easy", "medium", "hard"]
    estimated_time: Annotated[int, Ge(5), Le(60)] = Field(description="Время в минутах")
    hint: Optional[str] = None

class ExerciseSet(BaseModel):
    """SGR Cycle: создание множественных заданий"""
    exercises: Annotated[List[Exercise], MinLen(5), MaxLen(12)] = Field(
        description="Набор разнообразных заданий"
    )
    total_time: int = Field(description="Общее время выполнения всех заданий")
    distribution_by_difficulty: Dict[str, int] = Field(
        description="Количество заданий каждой сложности"
    )

# 5. Quality Reviewer - Cascade Pattern  
class QualityMetrics(BaseModel):
    """SGR Cascade: пошаговая оценка качества"""
    # Шаг 1: Оценка полноты
    completeness_score: Annotated[int, Ge(1), Le(10)] = Field(
        description="Насколько полно покрыта тема"
    )
    
    # Шаг 2: Оценка ясности
    clarity_score: Annotated[int, Ge(1), Le(10)] = Field(
        description="Насколько понятно изложение"
    )
    
    # Шаг 3: Оценка практичности
    practical_relevance: Annotated[int, Ge(1), Le(10)] = Field(
        description="Практическая ценность материала"
    )
    
    # Шаг 4: Предложения по улучшению
    improvement_suggestions: Annotated[List[str], MaxLen(5)] = Field(
        description="Конкретные рекомендации по улучшению"
    )
    
    # Шаг 5: Общая оценка
    overall_quality: Literal["excellent", "good", "needs_work", "poor"]

# 6. Coordinator - Routing Pattern
class ProcessingStep(BaseModel):
    step: Literal["research", "analyze", "create", "generate", "review", "finalize"]
    status: Literal["pending", "in_progress", "completed", "failed"]
    result_summary: Optional[str] = None
    timestamp: datetime = Field(default_factory=datetime.now)

class WorkflowState(BaseModel):
    """SGR Routing: управление потоком выполнения"""
    current_step: ProcessingStep
    completed_steps: List[ProcessingStep] = []
    final_output_ready: bool = False
    quality_threshold_met: bool = False

print(" Все SGR схемы определены!")
print("\n Статистика схем:")
print(f"- Research Agent: {len(ResearchOutput.__fields__)} полей")
print(f"- Complexity Analyzer: 3 возможных маршрута")  
print(f"- Content Creator: {len(LearningMaterial.__fields__)} полей")
print(f"- Exercise Generator: генерирует 5-12 заданий")
print(f"- Quality Reviewer: {len(QualityMetrics.__fields__)} критериев оценки")
print(f"- Coordinator: отслеживает 6 этапов процесса")


📋 Определяем SGR схемы для всех агентов системы
✅ Все SGR схемы определены!

📊 Статистика схем:
- Research Agent: 5 полей
- Complexity Analyzer: 3 возможных маршрута
- Content Creator: 4 полей
- Exercise Generator: генерирует 5-12 заданий
- Quality Reviewer: 5 критериев оценки
- Coordinator: отслеживает 6 этапов процесса


/var/folders/6c/j5sghwf51gx41g_055zlcs_m0000gq/T/ipykernel_18294/123912045.py:139: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"- Research Agent: {len(ResearchOutput.__fields__)} полей")
/var/folders/6c/j5sghwf51gx41g_055zlcs_m0000gq/T/ipykernel_18294/123912045.py:141: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"- Content Creator: {len(LearningMaterial.__fields__)} полей")
/var/folders/6c/j5sghwf51gx41g_055zlcs_m0000gq/T/ipykernel_18294/123912045.py:143: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. De

In [ ]:
# 5.4 Реализация агентов (симуляция SmolAgents)

if USE_REAL_LLM:
    print("Создаем мультиагентную систему с РЕАЛЬНЫМИ LLM")
    print(f"Модель: {DEFAULT_MODEL}")
else:
    print("Создаем мультиагентную систему")
    print("Для демонстрации используем моки вместо реальных LLM")

import random
from typing import Callable

class MockLLMAgent:
    """
    Агент, который может работать как с реальными LLM (через OpenRouter), так и с моками
    """
    def __init__(self, agent_id: str, role: str, output_schema: BaseModel):
        self.agent_id = agent_id
        self.role = role
        self.output_schema = output_schema
        
    def process(self, input_data: Any) -> BaseModel:
        """Обрабатывает данные через LLM или моки"""
        print(f"   {self.agent_id} обрабатывает: {str(input_data)[:50]}...")
        
        # Пытаемся использовать реальный LLM если доступен
        if USE_REAL_LLM:
            result = self._process_with_llm(input_data)
            if result is not None:
                return result
            # Если LLM вызов не удался, fallback на моки
            print(f"   [Fallback] Используем мок для {self.agent_id}")
        
        # Используем моки
        if self.role == "researcher":
            return self._mock_research_output(input_data)
        elif self.role == "analyzer":
            return self._mock_complexity_analysis(input_data)
        elif self.role == "creator":
            return self._mock_content_creation(input_data)
        elif self.role == "generator":
            return self._mock_exercise_generation(input_data)
        elif self.role == "reviewer":
            return self._mock_quality_review(input_data)
        else:
            raise ValueError(f"Unknown role: {self.role}")
    
    def _process_with_llm(self, input_data: Any) -> Optional[BaseModel]:
        """Обрабатывает данные через реальный LLM"""
        try:
            # Создаем промпт в зависимости от роли агента
            prompt = self._create_prompt(input_data)
            system_prompt = self._get_system_prompt()
            
            # Вызываем LLM с structured output
            result = call_llm_with_schema(
                prompt=prompt,
                response_schema=self.output_schema,
                system_prompt=system_prompt
            )
            
            return result
        except Exception as e:
            print(f"   Ошибка LLM вызова для {self.agent_id}: {e}")
            return None
    
    def _get_system_prompt(self) -> str:
        """Возвращает системный промпт для каждой роли"""
        prompts = {
            "researcher": """Вы - эксперт-исследователь, специализирующийся на анализе образовательных тем.
Ваша задача: провести тщательное исследование темы и предоставить структурированную информацию.
Будьте точны, объективны и полны в своем анализе.""",
            
            "analyzer": """Вы - аналитик сложности учебных материалов.
Ваша задача: определить уровень сложности темы и подобрать оптимальный подход к обучению.
Учитывайте предварительные знания, время изучения и целевую аудиторию.""",
            
            "creator": """Вы - создатель учебного контента с опытом педагогического дизайна.
Ваша задача: создать структурированный, понятный и практически ориентированный учебный материал.
Фокусируйтесь на ясности изложения и практической применимости.""",
            
            "generator": """Вы - разработчик учебных заданий и упражнений.
Ваша задача: создать разнообразные задания, проверяющие понимание материала.
Задания должны быть практичными, разной сложности и иметь четкие критерии.""",
            
            "reviewer": """Вы - эксперт по контролю качества образовательных материалов.
Ваша задача: объективно оценить качество материалов по нескольким критериям.
Будьте конструктивны в критике и предлагайте конкретные улучшения."""
        }
        return prompts.get(self.role, "Вы - полезный ассистент.")
    
    def _create_prompt(self, input_data: Any) -> str:
        """Создает промпт для LLM в зависимости от роли"""
        if self.role == "researcher":
            topic = str(input_data).replace("Тема:", "").strip() if isinstance(input_data, str) else str(input_data)
            return f"""Проведите исследование следующей темы: "{topic}"

Предоставьте:
1. Краткое описание темы (2-3 предложения)
2. Ключевые концепции, которые нужно изучить (5-15 концепций)
3. Признаки сложности темы
4. Релевантные источники информации для изучения (минимум 3)
5. Оценку качества проведенного исследования (1-10)

Будьте конкретны и практичны."""
            
        elif self.role == "analyzer":
            research = input_data
            return f"""Проанализируйте сложность следующей темы и определите подход к обучению.

Тема: {research.topic_summary}
Ключевые концепции: {', '.join(research.key_concepts)}
Признаки сложности: {', '.join(research.difficulty_indicators)}

Определите:
1. Уровень сложности (beginner/intermediate/advanced)
2. Примерное время изучения в часах
3. Необходимые предварительные знания
4. Целевую аудиторию
5. Рекомендуемый путь изучения

Будьте реалистичны в оценках."""
            
        elif self.role == "creator":
            research, complexity = input_data
            return f"""Создайте структурированный учебный материал.

Тема: {research.topic_summary}
Уровень: {complexity.complexity.level}
Ключевые концепции: {', '.join(research.key_concepts[:5])}

Создайте:
1. Структуру материала (outline) - 4-8 разделов
2. Детальное содержание для основных разделов
3. Практические примеры (2-5 примеров)
4. Ключевые выводы (3-7 takeaways)

Материал должен быть практичным и понятным для уровня {complexity.complexity.level}."""
            
        elif self.role == "generator":
            content = input_data
            return f"""Создайте набор учебных заданий и упражнений.

На основе следующего учебного материала:
Разделы: {', '.join(content.outline)}
Ключевые выводы: {', '.join(content.key_takeaways)}

Создайте 5-12 разнообразных заданий:
- Типы: multiple_choice, coding, essay, practical, analysis
- Сложность: easy, medium, hard
- Время выполнения: 5-60 минут
- Распределите задания по сложности равномерно

Задания должны проверять понимание и практическое применение."""
            
        elif self.role == "reviewer":
            content, exercises = input_data
            return f"""Проведите контроль качества учебных материалов.

Материалы для проверки:
- Структура: {len(content.outline)} разделов
- Примеры: {len(content.practical_examples)} примеров
- Задания: {len(exercises.exercises)} заданий

Оцените (по шкале 1-10):
1. Полноту покрытия темы (completeness_score)
2. Ясность изложения (clarity_score)
3. Практическую ценность (practical_relevance)

Предоставьте:
- Конкретные предложения по улучшению (до 5 пунктов)
- Общую оценку качества (excellent/good/needs_work/poor)

Будьте объективны и конструктивны."""
        
        return str(input_data)
    
    def _mock_research_output(self, topic: str) -> ResearchOutput:
        """Мок для Research Agent"""
        topic_name = str(topic).replace("Тема:", "").strip()
        
        # Разные результаты для разных тем
        if "блокчейн" in topic_name.lower() or "blockchain" in topic_name.lower():
            return ResearchOutput(
                topic_summary=f"Блокчейн - это распределенная технология записи данных, обеспечивающая прозрачность и безопасность транзакций без центрального управления.",
                key_concepts=["хеш-функции", "криптография", "консенсус", "смарт-контракты", "децентрализация", "майнинг", "цифровые подписи"],
                difficulty_indicators=["требует понимание криптографии", "математические концепции", "программирование"],
                information_sources=["bitcoin.org", "ethereum.org", "MIT OpenCourseWare Blockchain", "Coursera Blockchain Specialization"],
                research_confidence=9
            )
        else:
            return ResearchOutput(
                topic_summary=f"Обширная тема '{topic_name}' требует структурированного подхода к изучению и практического применения знаний.",
                key_concepts=["основы", "применение", "практика", "теория", "методы", "инструменты"],
                difficulty_indicators=["требует базовые знания", "практические навыки"],
                information_sources=["Wikipedia", "Khan Academy", "Coursera", "edX"],
                research_confidence=7
            )
    
    def _mock_complexity_analysis(self, research_data: ResearchOutput) -> ComplexityAnalysis:
        """Мок для Complexity Analyzer"""
        # Анализируем сложность на основе ключевых концепций
        num_concepts = len(research_data.key_concepts)
        has_math = any(word in " ".join(research_data.difficulty_indicators).lower() 
                      for word in ["математ", "крипт", "алгоритм", "формул"])
        
        if num_concepts > 6 or has_math:
            complexity = AdvancedLevel(
                level="advanced",
                estimated_hours=35,
                prerequisites=["математика", "программирование", "криптография"],
                learning_approach="Глубокое изучение + исследовательские проекты"
            )
            audience = ["студенты IT", "разработчики", "исследователи"]
        elif num_concepts > 4:
            complexity = IntermediateLevel(
                level="intermediate", 
                estimated_hours=15,
                prerequisites=["основы программирования"],
                learning_approach="Теория + практические задачи"
            )
            audience = ["студенты", "начинающие разработчики"]
        else:
            complexity = BeginnerLevel(
                level="beginner",
                estimated_hours=8,
                prerequisites=[],
                learning_approach="Пошаговое изучение с множеством примеров"
            )
            audience = ["все желающие", "студенты-новички"]
        
        return ComplexityAnalysis(
            complexity=complexity,
            target_audience=audience,
            learning_path=["изучение основ", "практические задания", "реальные проекты"]
        )
        
    def _mock_content_creation(self, inputs: tuple) -> LearningMaterial:
        """Мок для Content Creator"""
        research_data, complexity_data = inputs
        topic = research_data.topic_summary.split('.')[0]
        
        return LearningMaterial(
            outline=[
                "Введение и мотивация",
                "Основные концепции", 
                "Детальное изучение",
                "Практические применения",
                "Заключение и выводы"
            ],
            detailed_content={
                "Введение": f"Тема '{topic}' является важной областью современных технологий...",
                "Концепции": f"Ключевые понятия: {', '.join(research_data.key_concepts[:3])}...",
                "Применения": "Практическое использование в реальных проектах..."
            },
            practical_examples=[
                f"Пример 1: Базовое применение {research_data.key_concepts[0]}",
                f"Пример 2: Решение реальной задачи с {research_data.key_concepts[1]}",
                "Пример 3: Интеграция с существующими системами"
            ],
            key_takeaways=[
                f"Понимание {research_data.key_concepts[0]} критично для работы",
                "Практика важнее теории в данной области",
                "Постоянное обучение необходимо из-за быстрого развития"
            ]
        )
    
    def _mock_exercise_generation(self, content: LearningMaterial) -> ExerciseSet:
        """Мок для Exercise Generator"""
        exercises = [
            Exercise(
                type="multiple_choice",
                question=f"Что является ключевым в понимании {content.key_takeaways[0].split()[1]}?",
                difficulty="easy",
                estimated_time=10,
                hint="Вспомните основные концепции из введения"
            ),
            Exercise(
                type="practical",
                question="Реализуйте простой пример использования изученной технологии",
                difficulty="medium", 
                estimated_time=30,
                hint="Используйте один из практических примеров как основу"
            ),
            Exercise(
                type="analysis",
                question="Проанализируйте преимущества и недостатки подхода",
                difficulty="medium",
                estimated_time=20
            ),
            Exercise(
                type="coding",
                question="Напишите код, демонстрирующий основные принципы",
                difficulty="hard",
                estimated_time=45,
                hint="Объедините несколько концепций в одном решении"
            ),
            Exercise(
                type="essay", 
                question="Опишите возможности применения в вашей области",
                difficulty="medium",
                estimated_time=25
            )
        ]
        
        difficulty_count = {"easy": 1, "medium": 3, "hard": 1}
        total_time = sum(ex.estimated_time for ex in exercises)
        
        return ExerciseSet(
            exercises=exercises,
            total_time=total_time,
            distribution_by_difficulty=difficulty_count
        )
    
    def _mock_quality_review(self, materials: tuple) -> QualityMetrics:
        """Мок для Quality Reviewer"""
        content, exercises = materials
        
        # Оцениваем на основе контента
        completeness = 9 if len(content.outline) >= 5 else 7
        clarity = 8 if len(content.practical_examples) >= 3 else 6  
        practical = 9 if len(exercises.exercises) >= 5 else 7
        
        avg_score = (completeness + clarity + practical) / 3
        
        if avg_score >= 8.5:
            overall = "excellent"
        elif avg_score >= 7:
            overall = "good"
        else:
            overall = "needs_work"
        
        suggestions = []
        if completeness < 8:
            suggestions.append("Добавить больше деталей в содержание")
        if clarity < 8:
            suggestions.append("Улучшить ясность изложения")
        if practical < 8:
            suggestions.append("Добавить больше практических заданий")
        
        return QualityMetrics(
            completeness_score=completeness,
            clarity_score=clarity,
            practical_relevance=practical,
            improvement_suggestions=suggestions or ["Материал хорошего качества"],
            overall_quality=overall
        )

# Создаем всех агентов системы
agents = {
    "researcher": MockLLMAgent("research_agent_001", "researcher", ResearchOutput),
    "analyzer": MockLLMAgent("complexity_agent_001", "analyzer", ComplexityAnalysis),
    "creator": MockLLMAgent("content_agent_001", "creator", LearningMaterial),
    "generator": MockLLMAgent("exercise_agent_001", "generator", ExerciseSet),
    "reviewer": MockLLMAgent("quality_agent_001", "reviewer", QualityMetrics)
}

print(f" Создано {len(agents)} специализированных агентов")
for agent_id, agent in agents.items():
    print(f"    {agent.agent_id}: {agent.role}")


🤖 Создаем мультиагентную систему
📝 Для демонстрации используем моки вместо реальных LLM
✅ Создано 5 специализированных агентов
   🤖 research_agent_001: researcher
   🤖 complexity_agent_001: analyzer
   🤖 content_agent_001: creator
   🤖 exercise_agent_001: generator
   🤖 quality_agent_001: reviewer


In [ ]:
# 5.5 Координатор мультиагентной системы

class EducationalMaterialSystem:
    """
    Мультиагентная система для создания учебных материалов
    Демонстрирует все изученные паттерны взаимодействия
    """
    
    def __init__(self, agents: Dict[str, MockLLMAgent]):
        self.agents = agents
        self.workflow_state = None
        
    def create_educational_material(self, topic: str) -> Dict[str, Any]:
        """
        Основной метод создания учебных материалов
        Демонстрирует Hybrid Pattern: Hierarchical + Parallel + Sequential
        """
        print(f"\n СОЗДАНИЕ УЧЕБНЫХ МАТЕРИАЛОВ: {topic}")
        print("="*60)
        
        # Инициализация workflow
        self.workflow_state = WorkflowState(
            current_step=ProcessingStep(step="research", status="pending")
        )
        
        # ФАЗА 1: PARALLEL PATTERN - Исследование + Анализ сложности
        print("\n ФАЗА 1: Параллельный сбор информации")
        print("-" * 40)
        
        # Параллельно запускаем исследование (в реальности это было бы async)
        research_result = self._execute_step("research", topic)
        print(f" Исследование завершено (уверенность: {research_result.research_confidence}/10)")
        
        # Анализ сложности на основе результатов исследования  
        complexity_result = self._execute_step("analyze", research_result)
        complexity_level = complexity_result.complexity.level
        estimated_hours = complexity_result.complexity.estimated_hours
        print(f" Анализ сложности: {complexity_level} уровень ({estimated_hours}ч)")
        
        # ФАЗА 2: SEQUENTIAL PATTERN - Создание контента → Генерация заданий
        print("\n  ФАЗА 2: Последовательное создание контента")
        print("-" * 40)
        
        # Создание основного контента (использует результаты обеих предыдущих фаз)
        content_result = self._execute_step("create", (research_result, complexity_result))
        print(f" Контент создан ({len(content_result.outline)} разделов)")
        
        # Генерация упражнений (использует созданный контент)
        exercises_result = self._execute_step("generate", content_result) 
        print(f" Задания созданы ({len(exercises_result.exercises)} заданий, {exercises_result.total_time} мин)")
        
        # ФАЗА 3: REFLECTION PATTERN - Проверка качества + Итерации
        print("\n ФАЗА 3: Контроль качества и улучшения")
        print("-" * 40)
        
        # Проверка качества
        quality_result = self._execute_step("review", (content_result, exercises_result))
        print(f" Проверка качества: {quality_result.overall_quality}")
        print(f"    Баллы: полнота {quality_result.completeness_score}/10, "
              f"ясность {quality_result.clarity_score}/10, "
              f"практичность {quality_result.practical_relevance}/10")
        
        # Проверяем, нужны ли улучшения (Reflection Pattern)
        if quality_result.overall_quality in ["needs_work", "poor"]:
            print(" Требуются улучшения, запускаем итерацию...")
            # В реальной системе здесь был бы цикл улучшений
            print("   (В полной версии здесь был бы повторный цикл)")
        
        # Финализация
        self._mark_step_completed("finalize")
        self.workflow_state.final_output_ready = True
        self.workflow_state.quality_threshold_met = quality_result.overall_quality in ["good", "excellent"]
        
        print(f"\n СИСТЕМА ЗАВЕРШЕНА!")
        print(f"   Качество: {quality_result.overall_quality}")
        print(f"   Готов к использованию: {self.workflow_state.final_output_ready}")
        
        return {
            "topic": topic,
            "research": research_result,
            "complexity": complexity_result,
            "content": content_result,
            "exercises": exercises_result,
            "quality": quality_result,
            "workflow": self.workflow_state
        }
    
    def _execute_step(self, step_name: str, input_data: Any) -> Any:
        """Выполняет один шаг workflow через соответствующего агента"""
        
        # Обновляем состояние workflow (SGR Routing pattern)
        self._mark_step_in_progress(step_name)
        
        # Выбираем правильного агента для шага
        agent_mapping = {
            "research": "researcher",
            "analyze": "analyzer", 
            "create": "creator",
            "generate": "generator",
            "review": "reviewer"
        }
        
        if step_name not in agent_mapping:
            raise ValueError(f"Unknown step: {step_name}")
            
        agent = self.agents[agent_mapping[step_name]]
        result = agent.process(input_data)
        
        # Отмечаем шаг как завершенный
        self._mark_step_completed(step_name)
        
        return result
    
    def _mark_step_in_progress(self, step_name: str):
        """Обновляет состояние workflow"""
        self.workflow_state.current_step = ProcessingStep(
            step=step_name, 
            status="in_progress"
        )
    
    def _mark_step_completed(self, step_name: str):
        """Отмечает шаг как завершенный"""
        completed_step = ProcessingStep(
            step=step_name,
            status="completed", 
            result_summary=f"Шаг {step_name} успешно завершен"
        )
        self.workflow_state.completed_steps.append(completed_step)
        
    def get_workflow_status(self) -> Dict[str, Any]:
        """Возвращает текущий статус workflow"""
        return {
            "current_step": self.workflow_state.current_step.step,
            "completed_steps": [step.step for step in self.workflow_state.completed_steps],
            "progress": f"{len(self.workflow_state.completed_steps)}/6 шагов",
            "ready": self.workflow_state.final_output_ready
        }

# Создаем систему
educational_system = EducationalMaterialSystem(agents)

print(" Мультиагентная система создана!")
print(f"   Агентов в системе: {len(agents)}")

---

## ЗАДАНИЕ 3: Расширение мультиагентной системы

**Описание задачи:**
Расширьте существующую систему создания учебных материалов, добавив нового агента - **Code Example Generator** (Генератор примеров кода). Этот агент будет создавать практические примеры кода для учебных материалов.

**Требования:**

1. **Создайте SGR схему** для Code Example Generator используя **Cycle паттерн**:
   - Агент должен генерировать от 2 до 5 примеров кода
   - Каждый пример должен иметь: язык программирования, код, описание, уровень сложности
   
2. **Интегрируйте агента в MockLLMAgent**:
   - Добавьте метод `_mock_code_generation()` в класс `MockLLMAgent`
   - Метод должен принимать контент и возвращать набор примеров кода
   
3. **Добавьте агента в систему**:
   - Добавьте нового агента в словарь `agents`
   - Интегрируйте вызов агента в workflow (после Content Creator, перед Exercise Generator)
   
4. **Протестируйте**:
   - Запустите систему с новым агентом
   - Убедитесь, что примеры кода генерируются корректно

**Подсказка:** 
- Используйте `Annotated[List[CodeExample], MinLen(2), MaxLen(5)]` для Cycle паттерна
- Добавьте шаг "generate_code" в координатор

**Время выполнения:** 20-25 минут


In [ ]:
# РЕШЕНИЕ ЗАДАНИЯ 3

# Шаг 1: Создаем SGR схему с Cycle паттерном
class CodeExample(BaseModel):
    """Один пример кода"""
    language: Literal["python", "javascript", "java", "cpp", "go"]
    code: str = Field(description="Код примера")
    description: str = Field(description="Что демонстрирует пример")
    complexity: Literal["basic", "intermediate", "advanced"]
    explanation: str = Field(description="Пояснение к коду")

class CodeExampleSet(BaseModel):
    """SGR Cycle: генерация множественных примеров кода"""
    examples: Annotated[List[CodeExample], MinLen(2), MaxLen(5)] = Field(
        description="Набор примеров кода разной сложности"
    )
    total_examples: int = Field(description="Общее количество примеров")
    languages_used: List[str] = Field(description="Используемые языки программирования")

# Шаг 2: Расширяем MockLLMAgent новым методом
def mock_code_generation(self, content: LearningMaterial) -> CodeExampleSet:
    """Мок для Code Example Generator"""
    # Генерируем примеры кода на основе контента
    key_concept = content.key_takeaways[0].split()[1] if content.key_takeaways else "концепции"
    
    examples = [
        CodeExample(
            language="python",
            code=f'''# Базовый пример: {key_concept}
def example_function():
    """Демонстрация базового использования"""
    result = process_data()
    return result
    
example_function()''',
            description=f"Базовое использование {key_concept} в Python",
            complexity="basic",
            explanation="Простой пример, показывающий основные принципы"
        ),
        CodeExample(
            language="python",
            code=f'''# Продвинутый пример: {key_concept}
class AdvancedExample:
    def __init__(self, config):
        self.config = config
        
    def process(self, data):
        # Обработка с учетом конфигурации
        return self._advanced_logic(data)
    
    def _advanced_logic(self, data):
        # Сложная логика
        return processed_data

example = AdvancedExample(config)
result = example.process(data)''',
            description=f"Продвинутое применение {key_concept} с ООП",
            complexity="advanced",
            explanation="Демонстрация использования в реальном приложении"
        ),
        CodeExample(
            language="javascript",
            code=f'''// Пример на JavaScript: {key_concept}
async function processData(input) {{
    try {{
        const result = await fetchData(input);
        return transform(result);
    }} catch (error) {{
        console.error('Error:', error);
        return null;
    }}
}}

processData(input).then(result => console.log(result));''',
            description=f"Асинхронная обработка с {key_concept}",
            complexity="intermediate",
            explanation="Пример с async/await и обработкой ошибок"
        )
    ]
    
    return CodeExampleSet(
        examples=examples,
        total_examples=len(examples),
        languages_used=list(set(ex.language for ex in examples))
    )

# Добавляем метод в класс MockLLMAgent
MockLLMAgent._mock_code_generation = mock_code_generation

# Шаг 3: Добавляем нового агента в систему
code_generator_agent = MockLLMAgent("code_generator_001", "code_generator", CodeExampleSet)

# Обновляем словарь агентов
agents["code_generator"] = code_generator_agent

print("Новый агент Code Example Generator добавлен!")
print(f"Всего агентов в системе: {len(agents)}")

# Шаг 4: Обновляем класс EducationalMaterialSystem для использования нового агента
# (В реальной системе нужно было бы изменить метод create_educational_material)

# Демонстрация работы нового агента
print("\nДемонстрация работы Code Example Generator:")
print("-" * 50)

# Создаем тестовый контент
test_content = LearningMaterial(
    outline=["Введение", "Основы", "Практика"],
    detailed_content={"Введение": "Тестовый контент"},
    practical_examples=["Пример 1", "Пример 2"],
    key_takeaways=["Понимание алгоритмов критично", "Практика важна"]
)

# Генерируем примеры кода
code_examples = code_generator_agent.process(test_content)

print(f"\nСгенерировано примеров: {code_examples.total_examples}")
print(f"Используемые языки: {', '.join(code_examples.languages_used)}")

for i, example in enumerate(code_examples.examples, 1):
    print(f"\nПример {i}: [{example.language}] - {example.complexity}")
    print(f"Описание: {example.description}")
    print(f"Код:\n{example.code[:100]}...")  # Показываем первые 100 символов
    print(f"Пояснение: {example.explanation}")

print("\n" + "=" * 50)
print("Задание 3 выполнено!")
print("Cycle паттерн успешно применен для генерации множественных примеров кода")


# Демонстрация с реальным LLM (если доступен)
if USE_REAL_LLM:
    print("\n" + "=" * 50)
    print("БОНУС: Генерация примеров кода через LLM")
    print("=" * 50)
    
    prompt = f"""Создайте 3 примера кода для учебного материала по теме: {test_content.key_takeaways[0]}

Требования:
- 2-3 примера кода
- Разные языки программирования (python, javascript)
- Разная сложность (basic, intermediate, advanced)
- Каждый пример с описанием и пояснением

Форматируйте код правильно с отступами."""
    
    try:
        llm_code_examples = call_llm_with_schema(
            prompt=prompt,
            response_schema=CodeExampleSet,
            system_prompt="Вы - опытный преподаватель программирования, создающий учебные примеры кода."
        )
        
        if llm_code_examples:
            print(f"\nLLM сгенерировал {llm_code_examples.total_examples} примеров!")
            print(f"Языки: {', '.join(llm_code_examples.languages_used)}")
            
            for i, ex in enumerate(llm_code_examples.examples[:2], 1):
                print(f"\n[{i}] {ex.language} - {ex.complexity}")
                print(f"Описание: {ex.description}")
                print(f"Код (первые 150 символов):\n{ex.code[:150]}...")
    except Exception as e:
        print(f"Ошибка при вызове LLM: {e}")


In [11]:
# 5.6 Живая демонстрация системы

print(" ДЕМОНСТРАЦИЯ: Создание учебных материалов по блокчейну")
print("="*70)

# Запускаем систему с конкретной темой
result = educational_system.create_educational_material("Blockchain технологии")

# Показываем детальные результаты
print("\n ИТОГОВЫЕ РЕЗУЛЬТАТЫ:")
print("="*50)

print("\n 1. РЕЗУЛЬТАТЫ ИССЛЕДОВАНИЯ:")
research = result["research"]
print(f" Описание: {research.topic_summary}")
print(f" Ключевые концепции: {', '.join(research.key_concepts[:5])}...")
print(f" Сложность: {', '.join(research.difficulty_indicators)}")
print(f" Источники: {len(research.information_sources)} источников")
print(f" Уверенность: {research.research_confidence}/10")

print("\n 2. АНАЛИЗ СЛОЖНОСТИ:")
complexity = result["complexity"]
comp_info = complexity.complexity
print(f" Уровень: {comp_info.level}")
print(f"⏱ Время изучения: {comp_info.estimated_hours} часов")
print(f" Предварительные знания: {comp_info.prerequisites}")
print(f" Целевая аудитория: {', '.join(complexity.target_audience)}")
print(f" Подход к изучению: {comp_info.learning_approach}")

print("\n 3. СОЗДАННЫЙ КОНТЕНТ:")
content = result["content"]
print(f" Структура ({len(content.outline)} разделов):")
for i, section in enumerate(content.outline, 1):
    print(f"   {i}. {section}")

print(f"\n Практические примеры:")
for i, example in enumerate(content.practical_examples, 1):
    print(f"   {i}. {example}")

print(f"\n Ключевые выводы:")
for i, takeaway in enumerate(content.key_takeaways, 1):
    print(f"   {i}. {takeaway}")

print("\n 4. СОЗДАННЫЕ ЗАДАНИЯ:")
exercises = result["exercises"]
print(f" Всего заданий: {len(exercises.exercises)}")
print(f"⏱ Общее время: {exercises.total_time} минут")
print(f" Распределение: {exercises.distribution_by_difficulty}")

print(" Примеры заданий:")
for i, exercise in enumerate(exercises.exercises[:3], 1):
    print(f"   {i}. [{exercise.type}] {exercise.question}")
    print(f"      Сложность: {exercise.difficulty}, время: {exercise.estimated_time} мин")
    if exercise.hint:
        print(f"       Подсказка: {exercise.hint}")

print("\n 5. ОЦЕНКА КАЧЕСТВА:")
quality = result["quality"]
print(f" Полнота: {quality.completeness_score}/10")
print(f" Ясность: {quality.clarity_score}/10") 
print(f" Практичность: {quality.practical_relevance}/10")
print(f" Общая оценка: {quality.overall_quality}")
print(f" Рекомендации: {', '.join(quality.improvement_suggestions)}")

# Демонстрируем применение всех паттернов
print("\n ПРИМЕНЕННЫЕ ПАТТЕРНЫ:")
print("="*40)
print(" SGR Паттерны:")
print("    Cascade: Research Agent, Content Creator, Quality Reviewer")
print("    Routing: Complexity Analyzer, Coordinator") 
print("    Cycle: Exercise Generator")

print("\n Мультиагентные паттерны:")
print("    Parallel: Исследование + анализ сложности одновременно")
print("    Sequential: Создание контента → генерация заданий")
print("    Hierarchical: Координатор управляет специализированными агентами")
print("    Reflection: Проверка качества + возможность улучшений")

# Финальная статистика
workflow_status = educational_system.get_workflow_status()
print(f"\n СТАТИСТИКА ВЫПОЛНЕНИЯ:")
print(f"   Прогресс: {workflow_status['progress']}")
print(f"   Завершенные этапы: {', '.join(workflow_status['completed_steps'])}")
print(f"   Готов к использованию: {'Да' if workflow_status['ready'] else 'Нет'}")


🎬 ДЕМОНСТРАЦИЯ: Создание учебных материалов по блокчейну

🎯 СОЗДАНИЕ УЧЕБНЫХ МАТЕРИАЛОВ: Blockchain технологии

📚 ФАЗА 1: Параллельный сбор информации
----------------------------------------
  🔄 research_agent_001 обрабатывает: Blockchain технологии...
✅ Исследование завершено (уверенность: 9/10)
  🔄 complexity_agent_001 обрабатывает: topic_summary='Блокчейн - это распределенная техно...
✅ Анализ сложности: advanced уровень (35ч)

✍️  ФАЗА 2: Последовательное создание контента
----------------------------------------
  🔄 content_agent_001 обрабатывает: (ResearchOutput(topic_summary='Блокчейн - это расп...
✅ Контент создан (5 разделов)
  🔄 exercise_agent_001 обрабатывает: outline=['Введение и мотивация', 'Основные концепц...
✅ Задания созданы (5 заданий, 130 мин)

🔍 ФАЗА 3: Контроль качества и улучшения
----------------------------------------
  🔄 quality_agent_001 обрабатывает: (LearningMaterial(outline=['Введение и мотивация',...
✅ Проверка качества: excellent
   📊 Баллы: полнота 9/1

## Заключение и выводы

###  Что мы изучили на семинаре:

**1. Schema Guided Reasoning (SGR):**
-  **Cascade**: Структурированная последовательность рассуждений
-  **Routing**: Принуждение выбора одного пути из множества
-  **Cycle**: Создание множественных результатов одного типа

**2. Паттерны мультиагентных систем:**
-  **Sequential**: Пайплайн обработки данных
-  **Parallel**: Одновременное выполнение независимых задач  
-  **Hierarchical**: Координация через супервизора
-  **Reflection**: Итеративное улучшение качества

**3. Практическая реализация:**
-  Создали систему из 6 специализированных агентов
-  Применили все изученные паттерны в одном проекте
-  Продемонстрировали координацию и коммуникацию

###  Ключевые преимущества мультиагентных систем:

1. **Специализация**: Каждый агент экспертен в своей области
2. **Масштабируемость**: Легко добавлять новых агентов
3. **Отказоустойчивость**: Сбой одного агента не блокирует систему
4. **Производительность**: Параллельная обработка ускоряет работу
5. **Модульность**: Агенты можно переиспользовать в других системах


###  Ресурсы для дальнейшего изучения:

- **SmolAgents GitHub**: https://github.com/huggingface/smolagents
- **SGR Patterns**: https://abdullin.com/schema-guided-reasoning/patterns
- **CrewAI**: https://crewai.io/
- **AutoGen**: https://microsoft.github.io/autogen/
- **LangGraph**: https://langchain-ai.github.io/langgraph/

---

**Спасибо за участие в семинаре! **

*Вопросы и обсуждения приветствуются!*


---

## Использование OpenRouter API в заданиях

Все задания в этом notebook теперь поддерживают работу с **реальными LLM** через OpenRouter API!

### Какие задания используют LLM:

**ЗАДАНИЕ 1 - Translator Agent:**
- При наличии `OPENROUTER_API_KEY` использует LLM для перевода текста
- Демонстрирует Cascade паттерн в действии
- Fallback на моки если LLM недоступен

**ЗАДАНИЕ 3 - Code Example Generator:**
- БОНУС: Добавлен вызов LLM для генерации примеров кода
- Демонстрирует Cycle паттерн с реальной моделью

**ЗАДАНИЕ 4 - Code Review System:**
- Style Checker использует LLM для анализа стиля кода
- Security Analyzer использует LLM для поиска уязвимостей
- Автоматический fallback на моки

### Стоимость выполнения с LLM:

При использовании `anthropic/claude-3.5-sonnet`:
- Все задания: ~$0.06-0.10

**Моки полностью функциональны** - можно использовать их для обучения без затрат!


## Секция X:  мультиагентная система с инструментами и коммуникацией

В этом блоке мы реализуем реальную мультиагентную систему:
- Несколько агентов (Planner, Coder, Tester)
- Общаются через шину сообщений (message bus)
- Вызывают инструменты: запись и запуск кода, тестирование
- Используют OpenRouter (если доступен) для принятия решений и координации


In [20]:
# X.1 Инструменты и рабочее пространство (CodeWorkspace)

from types import SimpleNamespace
from typing import Dict, Any, List
import traceback
from io import StringIO
import sys
import difflib

class CodeWorkspace:
    """Простое рабочее пространство для хранения и выполнения кода."""
    def __init__(self):
        self.files: Dict[str, str] = {}
        self.last_output: str = ""
        self.last_error: str = ""
        self.context: Dict[str, Any] = {}  # общий контекст исполнения
    
    def store_code(self, filename: str, code: str) -> str:
        self.files[filename] = code
        return f"Stored {filename} ({len(code)} chars)"
    
    def read_code(self, filename: str) -> str:
        return self.files.get(filename, "")
    
    def run_python(self, code: str) -> Dict[str, str]:
        """Выполняет код в изолированном контексте и возвращает stdout / error."""
        captured_out = StringIO()
        old_stdout = sys.stdout
        sys.stdout = captured_out
        result = {"stdout": "", "error": ""}
        try:
            # Минимально изолированное окружение
            allowed_builtins = {
                "print": print, "len": len, "range": range, "enumerate": enumerate,
                "sum": sum, "min": min, "max": max, "abs": abs
            }
            exec_globals = {"__builtins__": allowed_builtins}
            exec_locals = self.context
            exec(code, exec_globals, exec_locals)
            result["stdout"] = captured_out.getvalue()
        except Exception as e:
            result["error"] = traceback.format_exc()
        finally:
            sys.stdout = old_stdout
        self.last_output = result["stdout"]
        self.last_error = result["error"]
        return result
    
    def run_tests(self, filename: str, tests_code: str) -> Dict[str, Any]:
        """Грузит код из файла + запускает тесты."""
        code = self.read_code(filename)
        if not code:
            return {"passed": False, "details": f"File {filename} not found"}
        # объединяем код и тесты
        combined = code + "\n\n" + tests_code
        res = self.run_python(combined)
        passed = (res["error"] == "")
        return {"passed": passed, "stdout": res["stdout"], "error": res["error"]}

workspace = CodeWorkspace()

# Реестр инструментов
class ToolError(Exception):
    pass

class ToolRegistry:
    def __init__(self, workspace: CodeWorkspace):
        self.ws = workspace
    
    def call(self, name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        # Нормализация аргументов и значения по умолчанию
        args = args or {}
        try:
            if name == "store_code":
                filename = args.get("filename", "solution.py")
                code = args.get("code", "")
                if not code:
                    return {"error": "store_code requires 'code'", "required": ["filename","code"]}
                prev = self.ws.read_code(filename)
                msg = self.ws.store_code(filename, code)
                diff = "\n".join(
                    difflib.unified_diff(
                        prev.splitlines(), code.splitlines(),
                        fromfile=f"prev:{filename}", tofile=f"new:{filename}", lineterm=""
                    )
                )
                head = code.splitlines()[:20]
                return {
                    "message": msg,
                    "filename": filename,
                    "chars": len(code),
                    "head": "\n".join(head),
                    "diff": diff[:4000]
                }
            if name == "read_code":
                filename = args.get("filename", "solution.py")
                return {"code": self.ws.read_code(filename)}
            if name == "run_python":
                code = args.get("code", "")
                if not code:
                    return {"error": "run_python requires 'code'", "required": ["code"]}
                return self.ws.run_python(code)
            if name == "run_tests":
                filename = args.get("filename", "solution.py")
                tests_code = args.get("tests_code", globals().get("tests", ""))
                if not tests_code:
                    return {"error": "run_tests requires 'tests_code'", "required": ["filename","tests_code"]}
                return self.ws.run_tests(filename, tests_code)
            return {"error": f"Unknown tool: {name}"}
        except Exception as e:
            return {"error": f"tool_exception: {type(e).__name__}: {e}"}

tools = ToolRegistry(workspace)
print("Tools ready: store_code, read_code, run_python, run_tests")


Tools ready: store_code, read_code, run_python, run_tests


In [21]:
# X.2 SGR: схема действий агента и сообщение

from pydantic import BaseModel
from typing import Union, Dict, Any, List, Literal

class AskAgent(BaseModel):
    action: str = "ask_agent"
    target: Literal["planner", "coder", "tester"]
    message: str

class UseTool(BaseModel):
    action: str = "use_tool"
    tool_name: Literal["store_code", "read_code", "run_python", "run_tests"]
    args: Dict[str, Any]

class Reply(BaseModel):
    action: str = "reply"
    content: str

class Finish(BaseModel):
    action: str = "finish"
    summary: str

class AgentAction(BaseModel):
    step: Union[AskAgent, UseTool, Reply, Finish]

class BusMessage(BaseModel):
    sender: Literal["planner", "coder", "tester"]
    recipient: Literal["planner", "coder", "tester", "broadcast"]
    content: str


In [24]:
# X.3 Реализация агентов и оркестратора

from collections import deque
from typing import List, Literal, Any

class MessageBus:
    def __init__(self):
        self.queue = deque()
        self.history: List[BusMessage] = []
    
    def send(self, msg: BusMessage):
        self.queue.append(msg)
        self.history.append(msg)
    
    def receive_for(self, agent_name: str) -> List[BusMessage]:
        messages = [m for m in list(self.queue) if m.recipient in (agent_name, "broadcast")]
        # remove delivered
        for m in messages:
            try:
                self.queue.remove(m)
            except ValueError:
                pass
        return messages

bus = MessageBus()

class LlmBackedAgent:
    def __init__(self, name: Literal["planner","coder","tester"], system_role: str):
        self.name = name
        self.system_role = system_role
    
    def decide(self, goal: str, bus_messages: List[BusMessage], allowed_tools: List[str]) -> AgentAction:
        """Запрашивает LLM и парсит в AgentAction с учётом доступных инструментов для роли."""
        # Собираем краткий контекст переписки
        history_text = "\n".join([f"{m.sender}→{m.recipient}: {m.content}" for m in bus_messages][-10:])
        allowed_str = ", ".join(allowed_tools) if allowed_tools else "(нет)"
        prompt = f"""
Вы — {self.system_role}. Ваша текущая роль: {self.name}.
Цель: {goal}
История сообщений (последние):
{history_text}

Выберите ОДНО действие:
- ask_agent: спросить другого агента (target in [planner,coder,tester]) с message
- use_tool: вызвать инструмент с tool_name и args (ДОСТУПНЫ ТОЛЬКО: {allowed_str})
- reply: отправить краткий ответ всем
- finish: когда цель достигнута, дать краткое резюме

Интерфейс инструментов (STRICT):
- store_code args: {{"filename": str (default: "solution.py"), "code": str (required)}}
- read_code  args: {{"filename": str (default: "solution.py")}}
- run_python args: {{"code": str (required)}}
- run_tests  args: {{"filename": str (default: "solution.py"), "tests_code": str (default: tests variable)}}

ОГРАНИЧЕНИЯ:
- Нельзя вызывать инструменты, которых нет в списке ДЛЯ ТЕКУЩЕЙ РОЛИ: {allowed_str}
- Если вы tester: после успешных тестов (passed=true) выберите finish с кратким резюме.
- Если инструмент вернул error, сформулируйте ask_agent к соответствующей роли с подробностями ошибки.

Верните ТОЛЬКО JSON строго по схеме AgentAction.
"""
        if not USE_REAL_LLM:
            raise RuntimeError("OPENROUTER_API_KEY is required: real multi-agent interaction runs without mocks only.")
        action = call_llm_with_schema(prompt=prompt, response_schema=AgentAction,
                                      system_prompt=self.system_role)
        if not action:
            raise RuntimeError("LLM did not return a valid AgentAction. Ensure the model supports JSON schema and API key is valid.")
        return action

class Orchestrator:
    def __init__(self):
        self.agents = {
            "planner": LlmBackedAgent("planner", "Планировщик, координирующий работу команды"),
            "coder": LlmBackedAgent("coder", "Разработчик, пишет код и использует инструменты выполнения кода"),
            "tester": LlmBackedAgent("tester", "Тестировщик, пишет и запускает тесты, сообщает о результатах")
        }
        # Разрешённые инструменты по ролям
        self.allowed_tools = {
            "planner": [],
            "coder": ["store_code", "read_code", "run_python"],
            "tester": ["read_code", "run_tests"]
        }
    
    def step(self, goal: str, turn_order: List[str]) -> bool:
        for name in turn_order:
            msgs = bus.receive_for(name)
            # Покажем входящие сообщения агенту
            if msgs:
                print(f"[{name}] inbox ({len(msgs)}):")
                for m in msgs[-3:]:
                    print(f"  {m.sender}→{m.recipient}: {m.content[:200]}")
            action = self.agents[name].decide(goal, bus.history, self.allowed_tools.get(name, []))
            # Трассировка принятого действия
            try:
                print(f"[{name}] action: {action.step.model_dump()}")
            except Exception:
                print(f"[{name}] action: {action}")
            step = action.step
            if isinstance(step, AskAgent):
                bus.send(BusMessage(sender=name, recipient=step.target, content=step.message))
                print(f"[{name}] → [{step.target}] ask: {step.message}")
            elif isinstance(step, UseTool):
                # Проверка прав на инструмент
                if step.tool_name not in self.allowed_tools.get(name, []):
                    msg = f"роль {name} не имеет доступа к инструменту {step.tool_name}. Доступны: {self.allowed_tools.get(name, [])}"
                    print(f"[{name}] denied {step.tool_name}: {msg}")
                    bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg))
                    continue
                result = tools.call(step.tool_name, step.args)
                # Подробный вывод результатов инструмента
                if step.tool_name == "store_code":
                    print(f"[{name}] used store_code on {result.get('filename')} ({result.get('chars')} chars)")
                    print("head:\n" + (result.get('head') or '')[:500])
                    if result.get('diff'):
                        print("diff:\n" + result['diff'][:1000])
                elif step.tool_name == "run_tests":
                    print(f"[{name}] used run_tests → passed={result.get('passed')}\nstdout:\n{(result.get('stdout') or '')[:800]}\nerror:\n{(result.get('error') or '')[:800]}")
                else:
                    print(f"[{name}] used {step.tool_name} → {str(result)[:200]}")
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"tool {step.tool_name} result: {result}"))
                # Авто-завершение при успешных тестах
                if step.tool_name == "run_tests" and isinstance(result, dict) and result.get("passed") is True:
                    summary = "Тесты пройдены. Цель достигнута."
                    bus.send(BusMessage(sender="tester", recipient="broadcast", content=f"FINISH: {summary}"))
                    print(f"[tester] finish: {summary}")
                    return True
            elif isinstance(step, Reply):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=step.content))
                print(f"[{name}] reply: {step.content}")
            elif isinstance(step, Finish):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"FINISH: {step.summary}"))
                print(f"[{name}] finish: {step.summary}")
                return True
        return False
    
    def run(self, goal: str, max_rounds: int = 8):
        print("=== ORCHESTRATION START ===")
        bus.send(BusMessage(sender="planner", recipient="broadcast", content=f"Начинаем: {goal}"))
        for r in range(1, max_rounds+1):
            print(f"\n--- ROUND {r} ---")
            finished = self.step(goal, ["planner", "coder", "tester"])  # фиксированный порядок
            if finished:
                print("=== ORCHESTRATION FINISHED ===")
                break
        else:
            print("=== MAX ROUNDS REACHED ===")


In [25]:
# X.4 Демонстрация: совместная реализация функции и прохождение тестов

goal = (
    "Реализовать функцию fib(n) -> int, возвращающую n-е число Фибоначчи (0-indexed), и пройти тесты.\n"
    "Ожидается простая и эффективная реализация без избыточной памяти."
)

# Предложение для агентов: Coder может вызвать use_tool/store_code для записи кода в file: solution.py
# Tester может вызвать use_tool/run_tests с тестами ниже

tests = r"""
# Тесты для функции fib (предполагают, что fib уже определена в загруженном коде)
# smoke tests
assert fib(0) == 0
assert fib(1) == 1
assert fib(5) == 5
assert fib(10) == 55
print("tests passed")
"""

orch = Orchestrator()

# Подскажем участникам стартовое сообщение с подсказкой про инструменты
bus.send(BusMessage(sender="planner", recipient="broadcast", content=(
    "Пожалуйста, Coder: используйте use_tool/store_code чтобы записать решение в solution.py \n"
    "Tester: используйте use_tool/run_tests с тестовым кодом. Если не проходят — сообщите и попросите доработку."
)))

orch.run(goal=goal, max_rounds=8)

print("\nWorkspace files:", list(workspace.files.keys()))
if "solution.py" in workspace.files:
    print("solution.py snippet:\n", workspace.files["solution.py"][:200])


=== ORCHESTRATION START ===

--- ROUND 1 ---
[planner] inbox (2):
  planner→broadcast: Пожалуйста, Coder: используйте use_tool/store_code чтобы записать решение в solution.py 
Tester: используйте use_tool/run_tests с тестовым кодом. Если не проходят — сообщите и попросите доработку.
  planner→broadcast: Начинаем: Реализовать функцию fib(n) -> int, возвращающую n-е число Фибоначчи (0-indexed), и пройти тесты.
Ожидается простая и эффективная реализация без избыточной памяти.
[planner] action: {'action': 'ask_agent', 'target': 'coder', 'message': 'Пожалуйста, реализуйте функцию fib(n) -> int, возвращающую n-е число Фибоначчи (0-indexed). Используйте итеративный подход для эффективности и минимального использования памяти.'}
[planner] → [coder] ask: Пожалуйста, реализуйте функцию fib(n) -> int, возвращающую n-е число Фибоначчи (0-indexed). Используйте итеративный подход для эффективности и минимального использования памяти.
[coder] inbox (1):
  planner→coder: Пожалуйста, реализуйте функцию